In [20]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

In [21]:
IMPORTANCE_THRESHOLD_LOGREG = 0.01

IMPORTANCE_THRESHOLD_CATBOOST = 0.005

RANDOM_STATE = 42

N_ESTIMATORS = 200

CLASS_WEIGHT = "balanced"

In [22]:
X_train = pd.read_pickle("../data/processed/X_train.pkl")
X_val = pd.read_pickle("../data/processed/X_val.pkl")
X_test = pd.read_pickle("../data/processed/X_test.pkl")

y_train = pd.read_pickle("../data/processed/y_train.pkl")
y_val = pd.read_pickle("../data/processed/y_val.pkl")
y_test = pd.read_pickle("../data/processed/y_test.pkl")

### Важность признаков

Для линейной модели также удаляем признак `R` и оставляем `number_inpatient`, поскольку он более интерпретируемый. А для ансамблиевой модели оставляем `R` вместо `number_inpatient`

In [23]:
rf_logreg = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)

X_train_logreg = X_train.copy().drop(columns=["R"])

rf_logreg.fit(X=X_train_logreg, y=y_train)

rf_importance_logreg = pd.Series(rf_logreg.feature_importances_, index=X_train_logreg.columns).sort_values(ascending=False)

Выбор признаков для линейной модели

In [24]:
selected_features_logreg = rf_importance_logreg[rf_importance_logreg > IMPORTANCE_THRESHOLD_LOGREG].index.tolist()

importance_df = pd.DataFrame(
    {
        "feature": X_train_logreg.columns,
        "importance": rf_logreg.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_df.head(len(selected_features_logreg))

,feature,importance
159,labs_per_day,0.063032
3,num_lab_procedures,0.058073
157,meds_per_day,0.057769
5,num_medications,0.050336
8,number_inpatient,0.046468
1,age,0.034241
158,procedures_per_day,0.033675
2,time_in_hospital,0.033351
9,number_diagnoses,0.028589
4,num_procedures,0.023192


In [25]:
print(f"Отобрано {len(selected_features_logreg)} признаков по Random Forest")

X_train_logreg[selected_features_logreg]

Отобрано 23 признаков по Random Forest


,labs_per_day,num_lab_procedures,meds_per_day,num_medications,number_inpatient,age,procedures_per_day,time_in_hospital,number_diagnoses,num_procedures,...,number_emergency,diag_3_Circulatory,diag_2_Circulatory,medical_specialty_Other,race_Caucasian,payer_code_Other,metformin,diag_1_Circulatory,admission_source_id_7,payer_code_MC
0,5.500000,-1.269231,6.500000,-0.2,1.0,2,2.500000,-0.50,-0.666667,2.0,...,0.0,0,0,1,0,1,0,0,1,0
1,22.000000,0.000000,8.000000,0.1,0.0,3,0.500000,-0.50,-0.333333,0.0,...,0.0,1,0,1,1,1,0,0,1,0
2,51.000000,0.269231,8.000000,-0.7,0.0,4,0.000000,-0.75,-1.000000,-0.5,...,0.0,0,0,1,1,1,0,0,1,0
3,10.333333,-0.500000,5.333333,0.1,0.0,5,2.000000,-0.25,0.333333,2.5,...,0.0,0,1,1,1,1,0,1,0,0
4,5.230769,0.923077,2.153846,1.3,0.0,8,0.153846,2.25,0.000000,0.5,...,0.0,0,1,1,1,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61239,4.200000,-0.884615,3.200000,0.1,1.0,7,0.200000,0.25,0.333333,0.0,...,0.0,0,0,1,1,1,0,0,1,0
61240,1.000000,-1.653846,15.000000,0.0,0.0,8,0.000000,-0.75,-0.333333,-0.5,...,0.0,0,0,1,1,0,0,1,1,1
61241,6.600000,-0.423077,3.600000,0.3,1.0,8,0.600000,0.25,0.333333,1.0,...,0.0,0,0,1,0,0,0,0,0,1
61242,4.500000,0.038462,2.100000,0.6,1.0,8,0.200000,1.50,0.333333,0.5,...,0.0,0,0,0,1,0,0,0,1,1


Выбор признаков для ансамблиевой модели

In [26]:
rf_catboost = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE, class_weight=CLASS_WEIGHT)

X_train_catboost = X_train.copy().drop(columns=["number_inpatient"])

rf_catboost.fit(X=X_train_catboost, y=y_train)

rf_importance_catboost = pd.Series(rf_catboost.feature_importances_, index=X_train_catboost.columns).sort_values(ascending=False)

In [27]:
selected_features_catboost = rf_importance_catboost[rf_importance_catboost > IMPORTANCE_THRESHOLD_CATBOOST].index.tolist()

importance_df = pd.DataFrame(
    {
        "feature": X_train_catboost.columns,
        "importance": rf_catboost.feature_importances_,
    }
).sort_values("importance", ascending=False)

importance_df.head(len(selected_features_catboost))

,feature,importance
158,labs_per_day,0.062811
3,num_lab_procedures,0.058044
156,meds_per_day,0.057217
161,R,0.055985
5,num_medications,0.049907
1,age,0.033914
157,procedures_per_day,0.033418
2,time_in_hospital,0.033212
8,number_diagnoses,0.028126
4,num_procedures,0.022849


In [28]:
print(f"Отобрано {len(selected_features_catboost)} признаков по Random Forest")

X_train_catboost[selected_features_catboost]

Отобрано 54 признаков по Random Forest


,labs_per_day,num_lab_procedures,meds_per_day,R,num_medications,age,procedures_per_day,time_in_hospital,number_diagnoses,num_procedures,...,high_medication_burden,diag_3_Respiratory,pioglitazone,medical_specialty_Family/GeneralPractice,diag_3_Genitourinary,rosiglitazone,diag_1_Injury,diag_1_Diabetes,diag_1_Symptoms,medical_specialty_Emergency/Trauma
0,5.500000,-1.269231,6.500000,0.4,-0.2,2,2.500000,-0.50,-0.666667,2.0,...,0,0,0,0,0,0,0,0,0,0
1,22.000000,0.000000,8.000000,0.2,0.1,3,0.500000,-0.50,-0.333333,0.0,...,1,0,0,0,0,0,0,0,0,0
2,51.000000,0.269231,8.000000,0.0,-0.7,4,0.000000,-0.75,-1.000000,-0.5,...,0,0,0,0,0,0,0,0,0,0
3,10.333333,-0.500000,5.333333,0.2,0.1,5,2.000000,-0.25,0.333333,2.5,...,1,0,0,0,0,0,0,0,0,0
4,5.230769,0.923077,2.153846,0.2,1.3,8,0.153846,2.25,0.000000,0.5,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61239,4.200000,-0.884615,3.200000,0.6,0.1,7,0.200000,0.25,0.333333,0.0,...,1,1,0,0,0,0,0,0,0,0
61240,1.000000,-1.653846,15.000000,0.0,0.0,8,0.000000,-0.75,-0.333333,-0.5,...,0,0,0,0,0,0,0,0,0,0
61241,6.600000,-0.423077,3.600000,0.6,0.3,8,0.600000,0.25,0.333333,1.0,...,1,0,0,0,0,0,0,0,0,0
61242,4.500000,0.038462,2.100000,0.6,0.6,8,0.200000,1.50,0.333333,0.5,...,1,0,1,0,0,0,1,0,0,0


Обучен Random Forest для оценки важности признаков. Выводы:

- feature engineering оправдан. labs_per_day, meds_per_day, procedures_per_day вошли в топ

- diag_*_Circulatory (сопутствующие болезни кровообращения) важнее diag_1_Circulatory (основного диагноза): в топ вошли diag_2_Circulatory и diag_3_Circulatory. Пациент с диабетом и сопутствующими сердечно-сосудистыми заболеваниями вероятнее обратится в больницу снова.

- демографические и административные признаки (race_Caucasian, payer_code_Other, admission_source_id_7) - в конце топа, вклад слабый (0.01). Ожидаемо: они косвенно связаны с тяжестью пациента, но не являются прямыми предикторами.

In [29]:
X_train_logreg[selected_features_logreg].to_pickle("../data/processed/X_train_selected_logreg.pkl")
X_val[selected_features_logreg].to_pickle("../data/processed/X_val_selected_logreg.pkl")
X_test[selected_features_logreg].to_pickle("../data/processed/X_test_selected_logreg.pkl")

X_train_catboost[selected_features_catboost].to_pickle("../data/processed/X_train_selected_catboost.pkl")
X_val[selected_features_catboost].to_pickle("../data/processed/X_val_selected_catboost.pkl")
X_test[selected_features_catboost].to_pickle("../data/processed/X_test_selected_catboost.pkl")